### Import

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from matplotlib import pyplot as plt 

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
eicu  = "PATH TO DATA/eICU/"
output  = "../Data/csvExtract/"

### Read icd diagnosis

In [3]:
diagnose = pd.read_csv(eicu + "diagnosis.csv")
diagnose = diagnose[['patientunitstayid', 'diagnosisoffset', 'icd9code']]

In [4]:
diagnose = diagnose[diagnose.icd9code.notnull()]
diagnose = diagnose.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True])
diagnose['icdCount'] = diagnose.icd9code.apply(lambda x: len(x.split(',')))
diagnose[['a', 'b', 'c', 'd', 'e', 'f', 'g']] = diagnose['icd9code'].str.split(pat=",", expand=True, n=-1)
diagnose.drop(columns=['icd9code', 'icdCount'], inplace=True)

In [5]:
diagnose['a'] = diagnose['a'].str.replace(' ', '')
diagnose['b'] = diagnose['b'].str.replace(' ', '')
diagnose['c'] = diagnose['c'].str.replace(' ', '')
diagnose['d'] = diagnose['d'].str.replace(' ', '')
diagnose['e'] = diagnose['e'].str.replace(' ', '')
diagnose['f'] = diagnose['f'].str.replace(' ', '')
diagnose['g'] = diagnose['g'].str.replace(' ', '')

diagnose = pd.melt(diagnose, id_vars=['patientunitstayid', 'diagnosisoffset'], 
                   value_vars=['a', 'b', 'c', 'd', 'e', 'f', 'g'], var_name='myVarname', value_name='icd9_code')

In [6]:
diagnose.drop(columns=['myVarname'], inplace=True)
diagnose = diagnose[diagnose.icd9_code.notnull()]
diagnose['icd10_code'] = diagnose.icd9_code
diagnose.loc[~(diagnose['icd10_code'].str.match('^[A-Z].*') == True), 'icd10_code'] = np.nan
diagnose.loc[diagnose.icd10_code.notnull(), 'icd9_code'] = np.nan
diagnose = diagnose.sort_values(['patientunitstayid', 'diagnosisoffset'], ascending=[True, True]).reset_index(drop=True)
diagnose['icd9_code']  = diagnose['icd9_code'].str.replace('.', '', regex=False)
diagnose['icd10_code'] = diagnose['icd10_code'].str.replace('.', '', regex=False)

In [8]:
diagnose.head(3)

### Congestive Heart Failure

In [9]:
diagnose['congestive_heart_failure'] = 0

diagnose.loc[diagnose.icd9_code.isin(['39891','40201','40211','40291','40401','40403','40411','40413',
                                      '40491','40493']) , 'congestive_heart_failure'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['4254','4255','4257',
                                                               '4258','4259']))) , 'congestive_heart_failure'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['428']))) , 
                                                                                   'congestive_heart_failure'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I099', 'I110', 'I130', 
      'I132', 'I255', 'I420', 'I425', 'I426', 'I427', 'I428', 'I429', 'P290']))) , 'congestive_heart_failure'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I43', 'I50']))) , 
                                                                                   'congestive_heart_failure'] = 1

### Cardiac Arrhythmias

In [10]:
diagnose['cardiac_arrhythmias'] = 0 

diagnose.loc[diagnose.icd9_code.isin(['42613','42610','42612','99601','99604']) , 'cardiac_arrhythmias'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['4260','4267','4269',
'4270','4271','4272','4273','4274','4276','4278','4279','7850','V450','V533']))) , 'cardiac_arrhythmias'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I441', 'I442', 'I443', 
             'I4566', 'I459', 'R000', 'R001', 'R008', 'T821', 'Z450', 'Z950']))) , 'cardiac_arrhythmias'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I47', 'I48', 'I49']))), 
                                                                                   'cardiac_arrhythmias'] = 1

### Valvular Disease

In [11]:
diagnose['valvular_disease'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['0932','7463','7464',
                                                      '7465','7466','V422','V433']))) ,  'valvular_disease'] = 1 

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['394','395','396','397',
                                                                              '424']))) , 'valvular_disease'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['A520', 'I091', 'I098', 
                             'Q230', 'Q231', 'Q232', 'Q233', 'Z952', 'Z953', 'Z954']))) , 'valvular_disease'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I05', 'I06', 'I07',
                                    'I08', 'I34', 'I35', 'I36', 'I37', 'I38', 'I39']))) , 'valvular_disease'] = 1

### Pulmonary Circulation

In [12]:
diagnose['pulmonary_circulation'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['4150','4151','4170',
                                                            '4178','4179']))) , 'pulmonary_circulation'] = 1 

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['416']))) , 
                                                                                'pulmonary_circulation'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I280', 'I288', 
                                                                   'I289']))) , 'pulmonary_circulation'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I26', 'I27']))) , 
                                                                                'pulmonary_circulation'] = 1

### Peripheral Vascular

In [13]:
diagnose['peripheral_vascular'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['0930','4373','4431',
                            '4432','4438','4439','4471','5571','5579','V434']))) , 'peripheral_vascular'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['440','441']))) , 
                                                                                   'peripheral_vascular'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I731', 'I738', 'I739',
              'I771', 'I790', 'I792', 'K551', 'K558', 'K559', 'Z958', 'Z959']))) , 'peripheral_vascular'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I70', 'I71']))) , 
                                                                                   'peripheral_vascular'] = 1

### Hypertension 

In [14]:
diagnose['hypertension'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['401']))) , 
                                                                                         'hypertension'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['402','403','404','405'])))
                                                                                        ,'hypertension'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I10']))) , 
                                                                                         'hypertension'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I11', 'I12', 'I13', 
                                                                             'I15']))) , 'hypertension'] = 1

### Paralysis

In [15]:
diagnose['paralysis'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['3341','3440','3441',
                                                '3442','3443','3444','3445','3446','3449']))) , 'paralysis'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['342','343']))) , 
                                                                                                'paralysis'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['G041', 'G114', 'G801',
                                   'G802', 'G830', 'G831', 'G832', 'G833', 'G834', 'G839']))) , 'paralysis'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['G81', 'G82']))) , 
                                                                                                'paralysis'] = 1

### Other Neurological

In [16]:
diagnose['other_neurological'] = 0

diagnose.loc[diagnose.icd9_code.isin(['33392']) , 'other_neurological'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['3319','3320','3321',
                                '3334','3335','3362','3481','3483','7803','7843']))) , 'other_neurological'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['334','335','340','341',
                                                                           '345']))) , 'other_neurological'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['G254', 'G255', 'G312',
                                          'G318', 'G319', 'G931', 'G934', 'R470']))) , 'other_neurological'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['G10', 'G11', 'G12',
     'G13', 'G20', 'G21', 'G22', 'G32', 'G35', 'G36', 'G37', 'G40', 'G41', 'R56']))) , 'other_neurological'] = 1

### Chronic Pulmonary

In [17]:
diagnose['chronic_pulmonary'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['4168','4169','5064',
                                                                    '5081','5088']))) , 'chronic_pulmonary'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['490','491','492','493',
                            '494','495','496','500','501','502','503','504','505']))) , 'chronic_pulmonary'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I278', 'I279', 'J684', 
                                                                   'J701', 'J703']))) , 'chronic_pulmonary'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['J40', 'J41', 'J42', 
'J43', 'J44', 'J45', 'J46', 'J47', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67']))),'chronic_pulmonary'] = 1

### Diabetes Uncomplicated

In [18]:
diagnose['diabetes_uncomplicated'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2500','2501','2502',
                                                                      '2503']))) , 'diabetes_uncomplicated'] = 1  

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E100', 'E101', 'E109', 
'E110', 'E111', 'E119', 'E120', 'E121', 'E129', 'E130', 'E131', 'E139', 'E140', 'E141', 'E149']))) , 
                                                                                   'diabetes_uncomplicated'] = 1

### Diabetes Complicated

In [19]:
diagnose['diabetes_complicated'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2504','2505','2506',
                                                          '2507','2508','2509']))) , 'diabetes_complicated'] = 1  

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E102', 'E103', 'E104',
'E105', 'E106', 'E107', 'E108', 'E112', 'E113', 'E114', 'E115', 'E116', 'E117', 'E118', 'E122', 'E123', 'E124', 
'E125', 'E126', 'E127', 'E128', 'E132', 'E133', 'E134', 'E135', 'E136', 'E137', 'E138', 'E142', 'E143', 'E144',
                                                'E145', 'E146', 'E147', 'E148']))) , 'diabetes_complicated'] = 1

### Hypothyroidism

In [20]:
diagnose['hypothyroidism'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2409','2461','2468']))), 
                                                                                           'hypothyroidism'] = 1  

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['243','244']))) , 
                                                                                           'hypothyroidism'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E890']))) , 
                                                                                           'hypothyroidism'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E0', 'E1', 'E2', 'E3']))),
                                                                                           'hypothyroidism'] = 1

### Renal Failure

In [21]:
diagnose['renal_failure'] = 0

diagnose.loc[diagnose.icd9_code.isin(['40301','40311','40391','40402','40403','40412','40413','40492','40493']) , 
                                                                                            'renal_failure'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['5880','V420','V451']))) , 
                                                                                            'renal_failure'] = 1  

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['585','586','V56']))) , 
                                                                                             'renal_failure'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I120', 'I131', 'N250', 
                                                'Z490', 'Z491', 'Z492', 'Z940', 'Z992']))) , 'renal_failure'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['N18', 'N19']))) , 
                                                                                             'renal_failure'] = 1

### Liver Disease

In [22]:
diagnose['liver_disease'] = 0

diagnose.loc[diagnose.icd9_code.isin(['07022','07023','07032','07033','07044','07054']) , 'liver_disease'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['0706','0709','4560',
       '4561','4562','5722','5723','5724','5728','5733','5734','5738','5739','V427']))) , 'liver_disease'] = 1 

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['570','571']))) ,
                                                                                          'liver_disease'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['I864', 'I982', 'K711',
'K713', 'K714', 'K715', 'K717', 'K760', 'K762', 'K763', 'K764', 'K765', 'K766', 'K767', 'K768', 'K769', 'Z944'])))
                                                                                        , 'liver_disease'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['B18', 'I85', 'K70', 
                                                                'K72', 'K73', 'K74']))) , 'liver_disease'] = 1

### Peptic Ulcer

In [23]:
diagnose['peptic_ulcer'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['5317','5319','5327',
                                                  '5329','5337','5339','5347','5349']))) , 'peptic_ulcer'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['K257', 'K259', 'K267', 
                                              'K269', 'K277', 'K279', 'K287', 'K289']))) , 'peptic_ulcer'] = 1

### Aids

In [24]:
diagnose['aids'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['042','043','044']))) , 
                                                                                                   'aids'] = 1 

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['B20', 'B21', 'B22', 
                                                                                       'B24']))) , 'aids'] = 1 

### Lymphoma

In [25]:
diagnose['lymphoma'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2030','2386']))) , 
                                                                                               'lymphoma'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['200','201','202']))) , 
                                                                                               'lymphoma'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['C900', 'C902']))) , 
                                                                                               'lymphoma'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['C81', 'C82', 'C83', 
                                                              'C84', 'C85', 'C88', 'C96']))) , 'lymphoma'] = 1

### Metastatic Cancer

In [26]:
diagnose['metastatic_cancer'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['196','197','198','199'])))
                                                                                    , 'metastatic_cancer'] = 1  

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['C77', 'C78', 'C79', 
                                                                          'C80']))) , 'metastatic_cancer'] = 1 

### Solid Tumor

In [27]:
diagnose['solid_tumor'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple([
         '140','141','142','143','144','145','146','147','148','149','150','151','152'
        ,'153','154','155','156','157','158','159','160','161','162','163','164','165'
        ,'166','167','168','169','170','171','172','174','175','176','177','178','179'
        ,'180','181','182','183','184','185','186','187','188','189','190','191','192'
        ,'193','194','195']))), 'solid_tumor'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple([
              'C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7'
            ,  'C8',  'C9', 'C10', 'C11', 'C12', 'C13', 'C14'
            , 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21'
            , 'C22', 'C23', 'C24', 'C25', 'C26', 'C30', 'C31'
            , 'C32', 'C33', 'C34', 'C37', 'C38', 'C39', 'C40'
            , 'C41', 'C43', 'C45', 'C46', 'C47', 'C48', 'C49'
            , 'C50', 'C51', 'C52', 'C53', 'C54', 'C55', 'C56'
            , 'C57', 'C58', 'C60', 'C61', 'C62', 'C63', 'C64'
            , 'C65', 'C66', 'C67', 'C68', 'C69', 'C70', 'C71'
            , 'C72', 'C73', 'C74', 'C75', 'C76', 'C97']))), 'solid_tumor'] = 1

### Rheumatoid Arthritis

In [28]:
diagnose['rheumatoid_arthritis'] = 0

diagnose.loc[diagnose.icd9_code.isin(['72889','72930']) , 'rheumatoid_arthritis'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['7010','7100','7101',
                        '7102','7103','7104','7108','7109','7112','7193','7285']))), 'rheumatoid_arthritis'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['446','714','720','725'])))
                                                                                   , 'rheumatoid_arthritis'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['L940', 'L941', 'L943', 
         'M120', 'M123', 'M310', 'M311', 'M312', 'M313', 'M461', 'M468', 'M469']))), 'rheumatoid_arthritis'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['M05', 'M06', 'M08', 
                                       'M30', 'M32', 'M33', 'M34', 'M35', 'M45']))), 'rheumatoid_arthritis'] = 1

### Coagulopathy

In [29]:
diagnose['coagulopathy'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2871','2873','2874',
                                                                                '2875']))), 'coagulopathy'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['286']))), 
                                                                                            'coagulopathy'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['D691', 'D693', 'D694',  
                                                                        'D695', 'D696']))), 'coagulopathy'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['D65', 'D66', 'D67', 
                                                                                  'D68']))), 'coagulopathy'] = 1

### Obesity

In [30]:
diagnose['obesity'] = 0

diagnose.loc[(diagnose.icd9_code.notnull())  & (diagnose.icd9_code.str.startswith(tuple(['2780']))), 'obesity'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E66']))), 'obesity'] = 1

### Weight Loss

In [31]:
diagnose['weight_loss'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['7832','7994']))), 
                                                                                             'weight_loss'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['260','261','262',
                                                                                  '263']))), 'weight_loss'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['R634', 'R64']))), 
                                                                                             'weight_loss'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E40', 'E41', 'E42', 
                                                             'E43', 'E44', 'E45', 'E46']))), 'weight_loss'] = 1

### Fluid Electrolyte

In [32]:
diagnose['fluid_electrolyte'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2536']))), 
                                                                                         'fluid_electrolyte'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['276']))), 
                                                                                         'fluid_electrolyte'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E222']))), 
                                                                                         'fluid_electrolyte'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['E86', 'E87']))), 
                                                                                         'fluid_electrolyte'] = 1

### Blood Loss Anemia

In [33]:
diagnose['blood_loss_anemia'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2800']))), 
                                                                                        'blood_loss_anemia'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['D500']))), 
                                                                                        'blood_loss_anemia'] = 1

### Deficiency Anemias

In [34]:
diagnose['deficiency_anemias'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2801','2808','2809']))), 
                                                                                        'deficiency_anemias'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['281']))), 
                                                                                        'deficiency_anemias'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['D508', 'D509']))), 
                                                                                        'deficiency_anemias'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['D51', 'D52', 'D53']))),
                                                                                        'deficiency_anemias'] = 1

### Alcohol Abuse

In [35]:
diagnose['alcohol_abuse'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2652','2911','2912',
'2913','2915','2918','2919','3030','3039','3050','3575','4255','5353','5710','5711','5712','5713','V113']))), 
                                                                                        'alcohol_abuse'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['980']))), 
                                                                                        'alcohol_abuse'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['F10', 'E52', 'G621', 
                    'I426', 'K292', 'K700', 'K703', 'K709', 'Z502', 'Z714', 'Z721']))), 'alcohol_abuse'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['T51']))), 
                                                                                        'alcohol_abuse'] = 1

### Drug Abuse

In [36]:
diagnose['drug_abuse'] = 0

diagnose.loc[diagnose.icd9_code.isin(['V6542']) , 'drug_abuse'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['3052','3053','3054',
                                                     '3055','3056','3057','3058','3059']))), 'drug_abuse'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['292','304']))), 
                                                                                             'drug_abuse'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['Z715', 'Z722']))), 
                                                                                             'drug_abuse'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['F11', 'F12', 'F13', 
                                                      'F14', 'F15', 'F16', 'F18', 'F19']))), 'drug_abuse'] = 1

### Psychoses

In [37]:
diagnose['psychoses'] = 0

diagnose.loc[diagnose.icd9_code.isin(['29604','29614','29644','29654']) , 'psychoses'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2938']))), 
                                                                                            'psychoses'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['295','297','298']))), 
                                                                                            'psychoses'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['F302', 'F312',
                                                                                'F315']))), 'psychoses'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['F20', 'F22', 'F23', 
                                                            'F24', 'F25', 'F28', 'F29']))), 'psychoses'] = 1

### Depression

In [38]:
diagnose['depression'] = 0

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['2962','2963','2965',
                                                                                '3004']))), 'depression'] = 1

diagnose.loc[(diagnose.icd9_code.notnull()) & (diagnose.icd9_code.str.startswith(tuple(['309','311']))), 
                                                                                            'depression'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['F204', 'F313', 'F314', 
                                                        'F315', 'F341', 'F412', 'F432']))), 'depression'] = 1

diagnose.loc[(diagnose.icd10_code.notnull()) & (diagnose.icd10_code.str.startswith(tuple(['F32', 'F33']))), 
                                                                                            'depression'] = 1

### Comorbidities

In [39]:
comorbidities = diagnose.drop(columns=['diagnosisoffset', 'icd9_code', 'icd10_code'])
comorbidities = comorbidities.groupby('patientunitstayid').max().reset_index()

### Final Data

In [40]:
comorbidities.head(3)

In [41]:
print(comorbidities.patientunitstayid.nunique())
print(comorbidities.shape)

155494
(155494, 31)


### Save Data

In [42]:
comorbidities.to_csv(output + 'comorbidities.csv', index=False)